In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt


def plot_latex_barchart(table_text, labels=None, figsize=None, title=""):
    lines = [l.strip() for l in table_text.splitlines() if l.strip()]
    pattern = r"([0-9]*\.[0-9]+)\s*\\pm\s*([0-9]*\.[0-9]+)"

    x_vals = []
    means, stds = [], []

    for line in lines:
        x = int(re.search(r"\d+", line).group())
        matches = re.findall(pattern, line)

        if not means:
            n = len(matches)
            means = [[] for _ in range(n)]
            stds = [[] for _ in range(n)]

        x_vals.append(x)

        for i, (m, s) in enumerate(matches):
            means[i].append(float(m))
            stds[i].append(float(s))

    if labels is None:
        labels = [f"Metric {i + 1}" for i in range(len(means))]

    # find row with maximum F1 (assumes column 2 is F1)
    f1_values = [row[2] for row in zip(*means)]  # transpose to get rows
    max_f1_index = np.argmax(f1_values)

    # nicer styling
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(len(x_vals))
    width = 0.8 / len(means)

    colors = ["#4C72B0", "#55A868", "#C44E52"]

    for i in range(len(means)):
        bar_colors = [colors[i]] * len(x_vals)
        # # highlight F1 bars
        # if i == 2:
        #     bar_colors[max_f1_index] = "#FFD700"  # gold for best F1

        ax.bar(
            x + i * width,
            means[i],
            width,
            yerr=stds[i],
            capsize=3,
            label=labels[i],
            color=bar_colors,
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_xticks(x + width * (len(means) - 1) / 2)
    ax.set_xticklabels(x_vals)
    ax.set_xlabel("# Retrieved Documents")
    ax.set_ylabel("Metrics")

    # cleaner axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    divider_x = x[0] + 0.765
    ax.axvline(divider_x, linestyle="--", linewidth=1, alpha=0.7)

    ax.legend(frameon=True, facecolor="white", framealpha=1)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.set_title(title)

    # compute bottom and top y-limits
    all_vals_lower = [
        m - s for metric, s_list in zip(means, stds) for m, s in zip(metric, s_list)
    ]
    all_vals_upper = [
        m + s for metric, s_list in zip(means, stds) for m, s in zip(metric, s_list)
    ]
    ymin = min(all_vals_lower) * 0.98
    ymax = max(all_vals_upper) * 1.02
    ax.set_ylim(bottom=ymin, top=ymax)

    plt.tight_layout()
    plt.show()

In [ ]:
import re


def f1_improvement_factor(table_str, column=2):
    """
    Compute improvement factor from the first row to the row with the maximum
    value in the specified metric column (default F1 = column 2).
    """

    # extract all mean ± std pairs
    pairs = re.findall(r"([0-9]*\.[0-9]+)\s*\\pm\s*([0-9]*\.[0-9]+)", table_str)

    # determine number of metrics per row
    n_metrics = 3
    rows = len(pairs) // n_metrics

    # reshape means into rows
    means = [float(p[0]) for p in pairs]
    means = [means[i * n_metrics : (i + 1) * n_metrics] for i in range(rows)]

    # first row
    first_value = means[0][column]

    # find row with max F1
    max_value = max(row[column] for row in means)

    # factor improvement
    factor = max_value / first_value

    # percentage improvement
    improvement_pct = (factor - 1) * 100

    return factor, improvement_pct

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt


def plot_multiple_f1_bars(
    table_strings,
    divider_pos,
    labels=None,
    figsize=(10, 5),
    title="",
    color_idx=0,
    color_palette=plt.cm.tab10.colors,
    plot_mean=False,
):
    """
    Plot multiple F1 curves as bars from multiple LaTeX-style table strings.

    Parameters
    ----------
    table_strings : list[str]
        List of LaTeX table strings (each with rows like '0 & $0.541 \pm 0.047$ ...')
    labels : list[str]
        Names for each series.
    figsize : tuple
        Figure size in inches.
    title : str
        Plot title.
    """
    n_series = len(table_strings)

    if labels is None:
        labels = [f"Series {i + 1}" for i in range(n_series)]

    n_rows = 0
    all_means, all_stds = [], []

    # parse each table
    for table in table_strings:
        lines = [l.strip() for l in table.splitlines() if l.strip()]
        pattern = r"([0-9]*\.[0-9]+)\s*\\pm\s*([0-9]*\.[0-9]+)"

        means, stds = [], []
        x_vals = []
        for line in lines:
            x = int(re.search(r"\d+", line).group())
            matches = re.findall(pattern, line)
            x_vals.append(x)
            if not means:
                n_metrics = len(matches)
                means = [[] for _ in range(n_metrics)]
                stds = [[] for _ in range(n_metrics)]
            for i, (m, s) in enumerate(matches):
                means[i].append(float(m))
                stds[i].append(float(s))
        if n_rows == 0:
            n_rows = len(means[0])
        all_means.append(means[2])  # F1 column only
        all_stds.append(stds[2])

    # plotting
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=figsize)

    x = np.arange(n_rows)
    width = 0.9 / n_series
    colors = color_palette

    for i in range(n_series):
        # highlight max F1 in the series
        # max_idx = np.argmax(all_means[i])
        bar_colors = [colors[color_idx + i % len(colors)]] * n_rows
        # bar_colors[max_idx] = "#FFD700"  # gold for max F1

        ax.bar(
            x + i * width,
            all_means[i],
            width,
            yerr=all_stds[i] if plot_mean else None,
            capsize=3,
            label=labels[i],
            color=bar_colors,
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_xticks(x + width * (n_series - 1) / 2)
    ax.set_xticklabels(range(n_rows))
    ax.set_xlabel("# Retrieved Documents")
    ax.set_ylabel("F1 Score")

    # vertical divider after the first group (optional)
    ax.axvline(divider_pos, linestyle="--", linewidth=1, alpha=0.7)

    # cleaner axes
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # y-axis limits with padding
    all_vals_lower = [
        m - s
        for series_m, series_s in zip(all_means, all_stds)
        for m, s in zip(series_m, series_s)
    ]
    all_vals_upper = [
        m + s
        for series_m, series_s in zip(all_means, all_stds)
        for m, s in zip(series_m, series_s)
    ]
    ax.set_ylim(bottom=min(all_vals_lower) * 0.98, top=max(all_vals_upper) * 1.02)

    ax.legend(frameon=True, facecolor="white", framealpha=1)
    ax.set_title(title)

    # pick a representative error bar (first series, middle x position works well visually)
    series_idx = 0
    bar_idx = len(all_means[0]) // 2

    x_pos = bar_idx + series_idx * width
    y_pos = all_means[series_idx][bar_idx]
    std = all_stds[series_idx][bar_idx]

    # ax.annotate(
    #     "tighter = more stable",
    #     xy=(x_pos, y_pos + std),  # arrow tip (top of error bar)
    #     xytext=(x_pos + 1.2, y_pos + std + 0.03),  # label position
    #     arrowprops=dict(arrowstyle="->", linewidth=1.2, color="black"),
    #     fontsize=12,
    #     ha="right",
    # )

    plt.tight_layout()
    plt.show()

In [ ]:
degasperi = r"""
0 & $0.541 \pm 0.047$ & $0.633 \pm 0.045$ & $0.583 \pm 0.044$ \\
1 & $0.598 \pm 0.059$ & $0.682 \pm 0.049$ & $0.637 \pm 0.053$ \\
2 & $0.650 \pm 0.058$ & $0.731 \pm 0.051$ & $0.688 \pm 0.055$ \\
3 & $0.667 \pm 0.051$ & $0.742 \pm 0.041$ & $0.702 \pm 0.046$ \\
4 & $0.683 \pm 0.053$ & $0.756 \pm 0.044$ & $0.717 \pm 0.049$ \\
5 & $0.698 \pm 0.044$ & $0.772 \pm 0.039$ & $0.733 \pm 0.041$ \\
6 & $0.712 \pm 0.045$ & $0.784 \pm 0.047$ & $0.746 \pm 0.045$ \\
\textbf{7} & $\boldsymbol{0.721 \pm 0.049}$ & $\boldsymbol{0.789 \pm 0.047}$ & $\boldsymbol{0.753 \pm 0.047}$ \\
"""

plot_latex_barchart(
    degasperi,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="De Marchi's Writings - Llama3.1 8B",
)


factor = f1_improvement_factor(degasperi)
factor, improvement_pct = f1_improvement_factor(degasperi)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
italiterature = r"""
0 & $0.693 \pm 0.046$ & $0.788 \pm 0.039$ & $0.737 \pm 0.044$ \\
1 & $0.775 \pm 0.054$ & $0.859 \pm 0.044$ & $0.815 \pm 0.051$ \\
2 & $0.822 \pm 0.055$ & $0.891 \pm 0.043$ & $0.855 \pm 0.051$ \\
3 & $0.841 \pm 0.050$ & $0.899 \pm 0.038$ & $0.869 \pm 0.047$ \\
4 & $0.853 \pm 0.049$ & $0.906 \pm 0.039$ & $0.878 \pm 0.046$ \\
5 & $0.859 \pm 0.050$ & $0.910 \pm 0.039$ & $0.884 \pm 0.047$ \\
6 & $0.867 \pm 0.051$ & $0.912 \pm 0.041$ & $0.889 \pm 0.048$ \\
\textbf{7} & $\boldsymbol{0.874 \pm 0.054}$ & $\boldsymbol{0.918 \pm 0.043}$ & $\boldsymbol{0.895 \pm 0.050}$ \\
"""

plot_latex_barchart(
    italiterature,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="Italian Literature - Llama3.1 8B",
)

factor = f1_improvement_factor(italiterature)
factor, improvement_pct = f1_improvement_factor(italiterature)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
aldomoro = r"""
0 & $0.693 \pm 0.050$ & $0.783 \pm 0.048$ & $0.735 \pm 0.049$ \\
1 & $0.763 \pm 0.047$ & $0.848 \pm 0.045$ & $0.803 \pm 0.046$ \\
2 & $0.807 \pm 0.035$ & $0.883 \pm 0.031$ & $0.843 \pm 0.033$ \\
3 & $0.834 \pm 0.033$ & $0.900 \pm 0.029$ & $0.866 \pm 0.032$ \\
4 & $0.852 \pm 0.027$ & $0.916 \pm 0.021$ & $0.883 \pm 0.025$ \\
5 & $0.861 \pm 0.024$ & $0.920 \pm 0.017$ & $0.890 \pm 0.021$ \\
6 & $0.875 \pm 0.023$ & $0.931 \pm 0.016$ & $0.902 \pm 0.020$ \\
\textbf{7} & $\boldsymbol{0.882 \pm 0.022}$ & $\boldsymbol{0.934 \pm 0.015}$ & $\boldsymbol{0.907 \pm 0.020}$ \\
"""

plot_latex_barchart(
    aldomoro,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="Carlo Visconti's Works - Llama3.1 8B",
)


factor = f1_improvement_factor(aldomoro)
factor, improvement_pct = f1_improvement_factor(aldomoro)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
wikinews = r"""
0 & $0.607 \pm 0.018$ & $0.704 \pm 0.022$ & $0.652 \pm 0.019$ \\
1 & $0.674 \pm 0.015$ & $0.759 \pm 0.019$ & $0.714 \pm 0.015$ \\
2 & $0.715 \pm 0.017$ & $0.787 \pm 0.017$ & $0.750 \pm 0.017$ \\
3 & $0.737 \pm 0.015$ & $0.801 \pm 0.018$ & $0.768 \pm 0.015$ \\
4 & $0.754 \pm 0.016$ & $0.812 \pm 0.017$ & $0.782 \pm 0.016$ \\
5 & $0.765 \pm 0.017$ & $0.821 \pm 0.018$ & $0.792 \pm 0.018$ \\
6 & $0.775 \pm 0.019$ & $0.831 \pm 0.019$ & $0.802 \pm 0.019$ \\
\textbf{7} & $\boldsymbol{0.781 \pm 0.014}$ & $\boldsymbol{0.836 \pm 0.016}$ & $\boldsymbol{0.807 \pm 0.014}$ \\
"""

plot_latex_barchart(
    wikinews,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="WikiNews - Llama3.1 8B",
)


factor = f1_improvement_factor(wikinews)
factor, improvement_pct = f1_improvement_factor(wikinews)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
sentences8b = """
0 & $0.776 \pm 0.073$ & $0.756 \pm 0.059$ & $0.765 \pm 0.065$ \\
1 & $0.824 \pm 0.073$ & $0.792 \pm 0.064$ & $0.807 \pm 0.068$ \\
2 & $0.819 \pm 0.057$ & $0.799 \pm 0.049$ & $0.808 \pm 0.052$ \\
3 & $0.837 \pm 0.033$ & $0.811 \pm 0.038$ & $0.823 \pm 0.035$ \\
4 & $0.841 \pm 0.036$ & $0.814 \pm 0.030$ & $0.828 \pm 0.033$ \\
5 & $0.839 \pm 0.038$ & $0.807 \pm 0.032$ & $0.823 \pm 0.034$ \\
6 & $0.844 \pm 0.045$ & $0.800 \pm 0.039$ & $0.821 \pm 0.040$ \\
\textbf{7} & $\boldsymbol{0.860 \pm 0.054}$ & $\boldsymbol{0.819 \pm 0.047}$ & $\boldsymbol{0.836 \pm 0.050}$ \\
"""
# 8 & $0.854 \pm 0.057$ & $0.822 \pm 0.051$ & $0.835 \pm 0.054$ \\

plot_latex_barchart(
    sentences8b,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="Juridical Sentences - Llama3.1 8B",
)


factor = f1_improvement_factor(sentences8b)
factor, improvement_pct = f1_improvement_factor(sentences8b)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
sentences70b = r"""
0 & $0.821 \pm 0.017$ & $0.844 \pm 0.019$ & $0.832 \pm 0.016$ \\
1 & $0.844 \pm 0.016$ & $0.865 \pm 0.017$ & $0.854 \pm 0.014$ \\
2 & $0.857 \pm 0.014$ & $0.872 \pm 0.020$ & $0.864 \pm 0.014$ \\
3 & $0.863 \pm 0.017$ & $0.875 \pm 0.022$ & $0.869 \pm 0.017$ \\
4 & $0.868 \pm 0.017$ & $0.878 \pm 0.021$ & $0.873 \pm 0.017$ \\
\textbf{5} & $\boldsymbol{0.873 \pm 0.020}$ & $\boldsymbol{0.876 \pm 0.027}$ & $\boldsymbol{0.875 \pm 0.022}$ \\
6 & $0.871 \pm 0.014$ & $0.874 \pm 0.026$ & $0.872 \pm 0.018$ \\
7 & $0.875 \pm 0.019$ & $0.869 \pm 0.035$ & $0.872 \pm 0.026$ \\
"""
# 8 & $0.873 \pm 0.015$ & $0.859 \pm 0.027$ & $0.866 \pm 0.019$ \\

plot_latex_barchart(
    sentences70b,
    labels=["Precision", "Recall", "F1"],
    figsize=(10, 4),
    title="Juridical Sentences - Llama3.1 70B",
)

factor = f1_improvement_factor(sentences70b)
factor, improvement_pct = f1_improvement_factor(sentences70b)
print(f"F1 improvement: {improvement_pct:.1f}%")

In [ ]:
plot_multiple_f1_bars(
    [
        degasperi,
        italiterature,
        aldomoro,
        wikinews,
        # sentences8b,
        # sentences70b,
    ],
    labels=[
        "De Marchi",
        "Italian Literature",
        "Carlo Visconti's Works",
        "WikiNews",
        # "Juridical Sentences (8B)",
        # "Juridical Sentences (70B)",
    ],
    figsize=(9, 6),
    title="F1 Comparison Across Models",
    divider_pos=0.835,
)

In [ ]:
plot_multiple_f1_bars(
    [
        sentences8b,
        sentences70b,
    ],
    labels=[
        "Juridical Sentences (8B)",
        "Juridical Sentences (70B)",
    ],
    figsize=(8, 5),
    title="Stability of F1-Score across Model Scales",
    divider_pos=0.730,
    color_palette=plt.cm.tab10.colors,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_f1_comparison():
    datasets = [
        # "De Marchi",
        # "Italian Literature",
        # "Carlo Visconti",
        # "WikiNews",
        "Juridical Sentences (8B)",
        "Juridical Sentences (70B)",
    ]

    # Only F1 values (3 per dataset)
    data = np.array(
        [
            # [0.936, 0.834, 0.733],
            # [0.927, 0.910, 0.953],
            # [0.973, 0.938, 0.900],
            # [0.922, 0.843, 0.816],
            [0.941, 0.712, 0.742],
            [0.943, 0.842, 0.820],
        ]
    )

    methods = ["PER", "LOC", "ORG"]

    x = np.arange(len(datasets))
    width = 0.25

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(7, 4))

    colors = plt.cm.Set2.colors

    for i in range(data.shape[1]):
        ax.bar(
            x + i * width,
            data[:, i],
            width,
            label=methods[i],
            color=colors[i],
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_xticks(x + width)
    ax.set_xticklabels(datasets)
    ax.set_title("Label-wise F1 across different model scales")

    ax.set_ylabel("F1 Score")
    ax.set_ylim(0.60, 1.0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(frameon=True, facecolor="white")
    plt.tight_layout()
    plt.show()


plot_f1_comparison()

In [ ]:
degasperi_proposed = "$0.721 \pm 0.049$ & $0.789 \pm 0.047$ & $0.753 \pm 0.047$ \\"
literature_proposed = "$0.874 \pm 0.054$ & $0.918 \pm 0.043$ & $0.895 \pm 0.050$ \\"
moro_proposed = "$0.882 \pm 0.022$ & $0.934 \pm 0.015$ & $0.907 \pm 0.020$ \\"
wikinews_proposed = "$0.781 \pm 0.014$ & $0.836 \pm 0.016$ & $0.807 \pm 0.014$"
juridical_proposed = "\textbf{5} & $\boldsymbol{0.873 \pm 0.020}$ & $\boldsymbol{0.876 \pm 0.027}$ & $\boldsymbol{0.875 \pm 0.022}$ \\"

degasperi_baseline = "0.842 & 0.828 & 0.835"
literature_baseline = "0.807 & 0.824 & 0.815"
moro_baseline = "0.948 & 0.966 & 0.957"
wikinews_baseline = "0.880 & 0.902 & 0.890"
juridical_baseline = "\textbf{0.915} & \textbf{0.920} & \textbf{0.917}"

plot_multiple_f1_bars(
    [
        degasperi,
        literature,
        moro,
        wikinews,
    ],
    labels=[
        "De Marchi's Writings",
        "Italian Literature",
        "Carlo Visconti's Works",
        "WikiNews",
        "Juridical Sentences (70B)",
    ],
    figsize=(8, 5),
    title="Stability of F1-Score across Model Scales",
    divider_pos=0.730,
    color_palette=plt.cm.tab10.colors,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

datasets = ["De Marchi", "Literature", "Moro", "WikiNews", "Juridical"]

proposed = [
    "$0.721 \pm 0.049$ & $0.789 \pm 0.047$ & $0.753 \pm 0.047$",
    "$0.874 \pm 0.054$ & $0.918 \pm 0.043$ & $0.895 \pm 0.050$",
    "$0.882 \pm 0.022$ & $0.934 \pm 0.015$ & $0.907 \pm 0.020$",
    "$0.781 \pm 0.014$ & $0.836 \pm 0.016$ & $0.807 \pm 0.014$",
    "\\textbf{5} & $0.873 \\pm 0.020$ & $0.876 \\pm 0.027$ & $0.875 \\pm 0.022$",
]

baseline = [
    "0.842 & 0.828 & 0.835",
    "0.807 & 0.824 & 0.815",
    "0.948 & 0.966 & 0.957",
    "0.880 & 0.902 & 0.890",
    "0.915 & 0.920 & 0.917",
]


def extract_proposed_f1(row):
    pattern = r"([0-9]*\.[0-9]+)\s*\\pm\s*([0-9]*\.[0-9]+)"
    matches = re.findall(pattern, row)
    f1_mean, f1_std = matches[-1]  # third metric
    return float(f1_mean), float(f1_std)


def extract_baseline_f1(row):
    return float(row.split("&")[-1])


proposed_means = []
proposed_stds = []
baseline_means = []

for p, b in zip(proposed, baseline):
    m, s = extract_proposed_f1(p)
    proposed_means.append(m)
    proposed_stds.append(s)
    baseline_means.append(extract_baseline_f1(b))

x = np.arange(len(datasets))
width = 0.35

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    baseline_means,
    width,
    label="Baseline",
    color=plt.cm.tab10.colors[0],
    edgecolor="black",
)

ax.bar(
    x + width / 2,
    proposed_means,
    width,
    capsize=4,
    label="Proposed",
    color=plt.cm.tab10.colors[8],
    edgecolor="black",
)

ax.set_xticks(x)
ax.set_xticklabels(datasets)
ax.set_ylabel("F1 Score")
ax.set_title("Baseline vs Proposed F1 across datasets")

ax.legend(frameon=True, facecolor="white")

plt.tight_layout()
plt.show()

In [ ]:
import re
import numpy as np


def parse_and_compare_stability(latex_8b: str, latex_70b: str) -> dict:
    """
    Parses LaTeX table rows of the form:
        k & $mean \\pm sigma$ & ... & $mean \\pm sigma$ \\
    and computes stability metrics comparing 8B vs 70B models.
    Uses the last column (F1) standard deviations.
    """

    def extract_sigmas(latex: str) -> np.ndarray:
        # Match all "mean ± sigma" pairs, take the last column per row
        row_pattern = re.compile(r"\\\\")
        sigma_pattern = re.compile(r"\$.*?\\pm\s*([\d.]+)\$")
        sigmas = []
        for row in latex.strip().split("\\\\"):
            matches = sigma_pattern.findall(row)
            if matches:
                sigmas.append(float(matches[-1]))  # last column = F1
        return np.array(sigmas)

    s8 = extract_sigmas(latex_8b)
    s70 = extract_sigmas(latex_70b)

    avg_8b, avg_70b = s8.mean(), s70.mean()
    max_8b, max_70b = s8.max(), s70.max()
    min_8b, min_70b = s8.min(), s70.min()

    return {
        "avg_sigma_8b": round(avg_8b, 4),
        "avg_sigma_70b": round(avg_70b, 4),
        "avg_sigma_reduction_%": round((avg_8b - avg_70b) / avg_8b * 100, 2),
        "max_sigma_8b": round(max_8b, 4),
        "max_sigma_70b": round(max_70b, 4),
        "max_sigma_reduction_%": round((max_8b - max_70b) / max_8b * 100, 2),
        "min_sigma_8b": round(min_8b, 4),
        "min_sigma_70b": round(min_70b, 4),
        "raw_sigmas_8b": s8.tolist(),
        "raw_sigmas_70b": s70.tolist(),
    }


latex_70b = r"""
0 & $0.821 \pm 0.017$ & $0.844 \pm 0.019$ & $0.832 \pm 0.016$ \\
1 & $0.844 \pm 0.016$ & $0.865 \pm 0.017$ & $0.854 \pm 0.014$ \\
2 & $0.857 \pm 0.014$ & $0.872 \pm 0.020$ & $0.864 \pm 0.014$ \\
3 & $0.863 \pm 0.017$ & $0.875 \pm 0.022$ & $0.869 \pm 0.017$ \\
4 & $0.868 \pm 0.017$ & $0.878 \pm 0.021$ & $0.873 \pm 0.017$ \\
\textbf{5} & $\boldsymbol{0.873 \pm 0.020}$ & $\boldsymbol{0.876 \pm 0.027}$ & $\boldsymbol{0.875 \pm 0.022}$ \\
6 & $0.871 \pm 0.014$ & $0.874 \pm 0.026$ & $0.872 \pm 0.018$ \\
7 & $0.875 \pm 0.019$ & $0.869 \pm 0.035$ & $0.872 \pm 0.026$ \\
8 & $0.873 \pm 0.015$ & $0.859 \pm 0.027$ & $0.866 \pm 0.019$ \\
"""

latex_8b = r"""
0 & $0.776 \pm 0.073$ & $0.756 \pm 0.059$ & $0.765 \pm 0.065$ \\
1 & $0.824 \pm 0.073$ & $0.792 \pm 0.064$ & $0.807 \pm 0.068$ \\
2 & $0.819 \pm 0.057$ & $0.799 \pm 0.049$ & $0.808 \pm 0.052$ \\
3 & $0.837 \pm 0.033$ & $0.811 \pm 0.038$ & $0.823 \pm 0.035$ \\
4 & $0.841 \pm 0.036$ & $0.814 \pm 0.030$ & $0.828 \pm 0.033$ \\
5 & $0.839 \pm 0.038$ & $0.807 \pm 0.032$ & $0.823 \pm 0.034$ \\
6 & $0.844 \pm 0.045$ & $0.800 \pm 0.039$ & $0.821 \pm 0.040$ \\
\textbf{7} & $\boldsymbol{0.860 \pm 0.054}$ & $\boldsymbol{0.819 \pm 0.047}$ &
$\boldsymbol{0.836 \pm 0.050}$ \\
8 & $0.854 \pm 0.057$ & $0.822 \pm 0.051$ & $0.835 \pm 0.054$ \\
"""

parse_and_compare_stability(latex_8b, latex_70b)